## PyTorch RNN Implementation

### Step 1 — Imports

In [92]:
import torch
import torch.nn as nn        # nn is a module in PyTorch that provides a wide range of functions and classes to build and train neural networks, including layers, activation functions, loss functions, and optimizers.
import numpy as np           # numpy is a library for numerical computing in Python, often used for handling arrays and performing mathematical operations. 

### Step 2 — Create a Toy Dataset for taining

We create a simple numeric sequence.

Example:
```
[1,2,3] → predict 4
[2,3,4] → predict 5

In [131]:
sequence_length = 3                  # 3 numbers in the sequence to predict the next number

X = []                               # empty list to store input sequences (3 numbers in each sequence)
y = []                               # empty list to store target values (the next number in the sequence)

data = np.arange(1,100)              # Creates array from 1 to 99, we will use this data to create sequences of 3 numbers and their corresponding next number as target for training the RNN.

for i in range(len(data)-sequence_length):             # Loop runs from i=0 to i=95 (96 iterations),len(data)-sequence_length because we need to stop at the point where we can still create a full sequence of 3 numbers and have a next number to predict, len(data) is 99 and sequence_length is 3, so we will iterate until index 96 (99-3) to create sequences like [1,2,3], [2,3,4], ..., [96,97,98] and their corresponding targets will be 4, 5, ..., 99 respectively.
    X.append(data[i:i+sequence_length])                # X.append(data[0:3]) → [1, 2, 3], at 0 index the value is 1. X is a list of sequences, each sequence is a list of 3 numbers. The loop creates sequences by slicing the data array from index i to i+sequence_length (i+3), so we get sequences like [1, 2, 3], [2, 3, 4], ..., [96, 97, 98].
    y.append(data[i+sequence_length])                  # y.append(data[3]) → 4, y.append(data[4]) → 5, ..., y.append(data[99]) → 100 (but we stop at index 96 for X, so the last target will be 99)

X = torch.tensor(X).float().unsqueeze(-1)              # Convert list of sequences to a PyTorch tensor and add an extra dimension at the end to represent the feature dimension (since we have only one feature, the numbers themselves). The shape of X will be (96, 3, 1) where 96 is the number of sequences, 3 is the sequence length, and 1 is the feature dimension. Feature means the number of input features for each time step in the sequence, in this case we have only one feature which is the number itself, so we add an extra dimension to make it compatible with the expected input shape for RNNs in PyTorch, which is (batch_size, sequence_length, input_size).         
y = torch.tensor(y).float()                            # Convert list of target values to a PyTorch tensor and convert to float for training the RNN. The shape of y will be (96,) which is a 1D tensor containing the target values corresponding to each sequence in X.

print(X.shape)                                          # (96, 3, 1), 1 means one number will be input at a time or time step from each sequence of 3 numbers, this is the input size for the RNN.
print(y.shape)                                          # (96,), 1D tensor containing the target values for each sequence in X, this is the output size for the RNN.

torch.Size([96, 3, 1])
torch.Size([96])



X = torch.tensor(X).float().unsqueeze(-1)
```
Step-by-step transformation:

1. torch.tensor(X): Convert list to tensor

Before: Python list of lists: [[1,2,3], [2,3,4], ...]
After:  Tensor shape: (96, 3) , 96 is the number of sequence and 3 is the number of lenght of each sequence

2. .float(): Convert to float type (needed for neural networks)
Before: torch.int64
After:  torch.float32


3. .unsqueeze(-1): Add a dimension at the end
Before shape: (96, 3)
After shape:  (96, 3, 1)
           batch, seq_len, features

Why unsqueeze? RNN expects input shape (batch, sequence_length, features)
Each number is a "feature" (just 1 feature per time step)

python
y = torch.tensor(y).float()
Convert targets to tensor:

Before: Python list: [4,5,6,...,99]
After:  Tensor shape: (96,)
Note: No unsqueeze needed here because MSELoss expects this shape

python
print(X.shape)
Output: torch.Size([96, 3, 1])

Interpretation:

96: Number of training samples

3: Sequence length (time steps)

1: Number of features at each step

### Step 3 — Build RNN Model from scratch

In [ ]:
class SimpleRNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.rnn = nn.RNN(
            input_size=1,                         # 1 number from each sequence at a time or time step
            hidden_size=16,                       # 16 numbers to represent the hidden state at each time step, it remebers the information from the previous time step and uses it to predict the next number
            batch_first=True                      # Input shape: (batch, seq, features), the sequence lenght is 3 then the output of rnn will be predicted based on the 3 numbers in the sequence, and the batch size is 96 because we have 96 sequences in our dataset.
        )
        
        self.fc = nn.Linear(16,1)                 # Fully connected layer to map the hidden state to the output, it takes the hidden state of size 16 and outputs a single number (the predicted next number in the sequence).

    def forward(self,x):
        
        out, hidden = self.rnn(x)                 # out shape: (batch_size(96), sequence_length(3), hidden_size(16)) → (96, 3, 16)-> 16 outputs from the RNN
        
        out = out[:,-1,:]                         # We take the output of the last time step (the last number in the sequence) for each sequence in the batch, which has the shape (batch_size, hidden_size) → (96, 16). This is because we want to use the information from the entire sequence to predict the next number, and the last time step's output contains the most relevant information for that prediction.
        
        out = self.fc(out)                        # Pass the output of the last time step through a fully connected layer to get the final prediction, which will have the shape (batch_size, output_size) → (96, 1) since we want to predict a single number for each sequence.
        
        return out

```
python
        self.rnn = nn.RNN(
            input_size=1,       [1 number from each sequence at a time]
            hidden_size=16,
            batch_first=True
        )
nn.RNN           Parameters      Explained:
Parameter	 Value	         Meaning
input_size	 1	         Each time step has 1 feature (just a number)
hidden_size	 16	         Memory size (16-dimensional hidden state)
batch_first	 True	         Input shape is (batch, seq_len, features)
 
What this RNN layer creates:
1. Weight matrices:

W_ih: For input → hidden (size: 16×1)

W_hh: For hidden → hidden (size: 16×16)

2. Bias vectors:

b_ih: For input (size: 16)

b_hh: For hidden (size: 16)

Total parameters:

Weights: (16×1) + (16×16) = 16 + 256 = 272

Biases: 16 + 16 = 32

Total: 304 parameters

How it processes data:
For each time step t:

h_t = tanh(W_ih · x_t + W_hh · h_{t-1} + b_ih + b_hh)
python
        self.fc = nn.Linear(16, 1)
Linear layer to convert RNN output to prediction:

Input features: 16 (the hidden state size)

Output features: 1 (single predicted number)

Parameters:

Weights: 16×1 = 16

Bias: 1

Total: 17 parameters

python
    def forward(self, x):
Forward pass definition: Called when we do model(X)

python
        out, hidden = self.rnn(x)
What happens inside RNN:
Input x shape: (batch_size, 3, 1)

Processing for one sample [a, b, c]:

Time t=1: 
   Input: a
   Hidden: h₁ = tanh(W·a + U·h₀)  # h₀ = zeros

Time t=2:
   Input: b
   Hidden: h₂ = tanh(W·b + U·h₁)

Time t=3:
   Input: c
   Hidden: h₃ = tanh(W·c + U·h₂)
   
Outputs:

out: All hidden states [h₁, h₂, h₃] shape: (batch, 3, 16)

hidden: Final hidden state h₃ shape: (1, batch, 16)

python
        out = out[:, -1, :]   # last timestep
What this does: Takes only the LAST hidden state

Before: out.shape = (batch, 3, 16)
After: out.shape = (batch, 16)

Why last timestep? For prediction, we only care about the final hidden state after seeing the entire sequence [a,b,c]. This contains all the information needed to predict the next number.

Index explanation:

: → take all batches

-1 → take the last time step (index 2)

: → take all 16 hidden features

python
        out = self.fc(out)
        return out
Final transformation:

out shape: (batch, 16)

Linear layer converts to (batch, 1)

Returns predictions for each sample

### COMPLETE VISUALIZATION: RNN Hidden States Explained
```
1. Let's Start with a Simple Analogy
Reading a Book with Notes
Imagine you're reading a mystery novel and taking notes:

As you read each page, you write down important clues in a notebook.
Your notebook has 16 pages (one for each type of clue).

Page 1: Suspect descriptions
Page 2: Locations mentioned  
Page 3: Time of events
Page 4: Weapons mentioned
...
Page 16: Motives

When you finish the book, you look at ALL 16 pages to guess who did it.
This is EXACTLY what RNN does!

Each page = one dimension of hidden state

16 pages = hidden_size=16

Final guess = prediction based on all 16 pages

2. What Does Hidden State Size 16 ACTUALLY Mean?
Visual Representation:

Hidden State at time t: [0.2, -0.5, 0.8, 0.1, -0.3, 0.6, ..., 0.4]
                         ↑     ↑     ↑    ↑     ↑    ↑         ↑
Dimension:               1     2     3    4     5    6        16

Each number is like a "light switch" that can be:
- Positive (>0): Feature is present/active
- Negative (<0): Feature is absent/inactive  
- Magnitude: How strong the feature is
What Each Dimension Might Represent (after training):

Dimension 1: "Increasing trend detector" → 0.8 means numbers are going UP
Dimension 2: "Decreasing trend detector" → -0.5 means NOT going down
Dimension 3: "Pattern repetition counter" → 0.1 means pattern might repeat
Dimension 4: "Anomaly detector" → -0.3 means no anomaly
...
Dimension 16: "Sequence length tracker" → 0.4 means we're in middle of sequence
The network LEARNS what each dimension should represent during training!

3. Let's Trace ONE Sequence Through the RNN
Input Sequence: [1, 2, 3]

Time Step 1: Input = 1
┌─────────────────────────────────────────────────────┐
│  Input (1)                                           │
│     ↓                                                │
│  ┌─────────────────────────────────────────────┐    │
│  │ RNN Cell at t=1                              │    │
│  │  h₁ = tanh(W·1 + U·h₀)  where h₀ = [0,...,0] │    │
│  └─────────────────────────────────────────────┘    │
│     ↓                                                │
│  Hidden State h₁ (16 numbers):                       │
│  [0.3, 0.1, -0.2, 0.5, 0.0, ..., 0.1]              │
│   ↑    ↑     ↑    ↑    ↑        ↑                    │
│   These 16 numbers encode: "I just saw number 1"    │
└─────────────────────────────────────────────────────┘

Time Step 2: Input = 2
┌─────────────────────────────────────────────────────┐
│  Input (2) + Previous Memory h₁                      │
│     ↓        ↓                                       │
│  ┌─────────────────────────────────────────────┐    │
│  │ RNN Cell at t=2                              │    │
│  │ h₂ = tanh(W·2 + U·h₁)                        │    │
│  └─────────────────────────────────────────────┘    │
│     ↓                                                │
│  Hidden State h₂ (16 numbers):                       │
│  [0.6, 0.3, 0.1, 0.2, -0.1, ..., 0.4]              │
│   ↑    ↑    ↑    ↑     ↑        ↑                    │
│   Now encodes: "I saw 1 then 2 (increasing)"        │
└─────────────────────────────────────────────────────┘

Time Step 3: Input = 3
┌─────────────────────────────────────────────────────┐
│  Input (3) + Previous Memory h₂                      │
│     ↓        ↓                                       │
│  ┌─────────────────────────────────────────────┐    │
│  │ RNN Cell at t=3                              │    │
│  │ h₃ = tanh(W·3 + U·h₂)                        │    │
│  └─────────────────────────────────────────────┘    │
│     ↓                                                │
│  Hidden State h₃ (16 numbers):                       │
│  [0.9, 0.5, 0.3, -0.1, 0.2, ..., 0.7]              │
│   ↑    ↑    ↑     ↑    ↑        ↑                    │
│   Encodes: "Complete pattern 1,2,3 (increasing)"    │
└─────────────────────────────────────────────────────┘
4. What Happens INSIDE an RNN Cell?
The Math Behind the Magic:

h_t = tanh( W · x_t + U · h_{t-1} + b )

Let's break this down for t=2:

h₂ = tanh( W·2 + U·h₁ + b )

Where:
W = [w₁, w₂, ..., w₁₆]ᵀ  (16×1 matrix)
     Each wᵢ is how much input affects dimension i

U = 16×16 matrix
     U[i,j] = how much previous state's jth dimension 
              affects current state's ith dimension

Example calculation for dimension 3 of h₂:

h₂[3] = tanh( W[3]×2 + Σ(U[3,j] × h₁[j]) + b[3] )
            ↑            ↑                       ↑
        New input    Memory from past         Bias
        influence    (forget/remember)
5. VISUALIZATION: The 16-Dimensional Hidden State

6. WHY 16? What if we use different sizes?
Hidden Size = 1 (Too Small)

Memory: [0.5]  ← Just one number to remember everything!

Problem: Can only remember one thing:
- Is it increasing? OR
- What was the last number? OR
- What's the pattern?
Can't capture all information → Poor predictions
Hidden Size = 16 (Good Balance)

Memory: [0.2, -0.5, 0.8, 0.1, -0.3, 0.6, ..., 0.4]

Can simultaneously remember:
- Trend (dim1): "increasing"
- Last value (dim2): "was 3"
- Pattern (dim3): "adds 1 each time"
- Position (dim4): "at step 3"
- Anomalies (dim5): "no anomalies"
- And 11 other features!
Hidden Size = 100 (Too Large)
Memory: 100 numbers

Problems:
- Overfitting (memorizes noise)
- Too many parameters (slow training)
- Needs more data
- Might remember irrelevant details
7. WHAT IF WE DON'T USE HIDDEN STATE?
Without Hidden State (Just Linear Layer):
# This would be just processing each input independently
def forward(self, x):
    # x shape: (batch, 3, 1)
    out = self.fc(x[:, -1, :])  # Only look at last number!
    return out
What this model learns:

Sees only the last number (3)

Predicts based ONLY on that number

Can't learn pattern "add 1"

Result: Predicts random numbers because 3 alone doesn't tell you what comes next!

Why We Need Hidden State:

Without Memory:
Input: [1,2,3] → Only sees "3" → Predicts ??? (no context)

With Memory:
Input: [1,2,3] → 
    Step1: "1" → h₁ remembers "started with 1"
    Step2: "2" → h₂ remembers "saw 1 then 2 (increasing)"
    Step3: "3" → h₃ remembers "consistent +1 pattern"
    Final: Based on h₃ → Predicts "4"
8. THE FLOW OF INFORMATION (VISUAL)

INPUT SEQUENCE:      [1]    →    [2]    →    [3]
                      │            │            │
                      ▼            ▼            ▼
HIDDEN STATE:      h₁[16]  →   h₂[16]  →   h₃[16]
                   (encodes 1)  (encodes   (encodes
                                1→2)       1→2→3)
                      │            │            │
                      └────────────┴────────────┘
                                  │
                                  ▼
                          FINAL PREDICTION: 4
                                  │
                                  ▼
                         Based on ALL history
                         stored in h₃[16]
9. ANALOGY: Detective's Notebook

Time Step 1 (See number 1):
┌─────────────────────────┐
│ Detective's Notes:      │
│ Page 1: "Saw number 1"  │
│ Page 2: "Start of seq"  │
│ Page 3: "???"           │
│ ... (16 pages total)    │
└─────────────────────────┘

Time Step 2 (See number 2):
┌─────────────────────────┐
│ Updated Notes:          │
│ Page 1: "Sequence: 1→2" │
│ Page 2: "Increasing"    │
│ Page 3: "Step 2 of 3"   │
│ ... (16 pages total)    │
└─────────────────────────┘

Time Step 3 (See number 3):
┌─────────────────────────┐
│ Final Notes:            │
│ Page 1: "Pattern: +1"   │
│ Page 2: "Complete: 1,2,3"│
│ Page 3: "Next should be 4"│
│ ... (16 pages total)    │
└─────────────────────────┘

Final Decision (Prediction):
Look at ALL 16 pages → Conclude next is 4!
10. SUMMARY: Hidden State Explained
Question	Answer
What is hidden state?	A 16-number vector that summarizes everything seen so far

Why 16 numbers?	Enough capacity to remember multiple aspects of the sequence

What do they store?	Patterns, trends, positions, relationships learned during training

Why not 1 number?	Can't remember multiple things simultaneously

Why not 100?	Overfitting, slow training, needs more data

How are they updated?	Each new input + old state → new state (like updating notes)

Where is the final one used?	Last hidden state (after all inputs) goes to Linear layer for prediction

Think of hidden state as the RNN's "working memory" - it's where all the magic of sequence understanding happens!

so does hidden state 16 means it has memory of 16 number or can memorise 16 inputs?
NO! This is a VERY Common Misconception! 
Hidden Size 16 Does NOT Mean "Remembers 16 Numbers"
What It IS:
Hidden size 16 = The RNN has 16 different "features" or "aspects" it tracks about the sequence

What It IS NOT:
NOT "remembers 16 previous inputs"
NOT "can store 16 numbers from the past"

Let's Use Better Analogies:
❌ WRONG Way to Think:

Hidden size 16 = Can remember 16 numbers: [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]
✅ CORRECT Way to Think:

Hidden size 16 = Has 16 different "perspectives" on the data:

Perspective 1: "Is the sequence increasing?" → 0.8 (yes, strongly)
Perspective 2: "What was the last value?" → 0.3 (medium)
Perspective 3: "Is there a pattern?" → 0.9 (yes, strongly)
Perspective 4: "How long is the sequence?" → 0.5 (medium)
Perspective 5: "Any anomalies?" → -0.2 (no)
... and 11 other perspectives!
Concrete Example with Numbers
Input Sequence: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
With Hidden Size = 16:

The RNN doesn't store these 20 numbers. Instead, it compresses them into 16 "summary statistics":
After each input those 16 summaries will be updated and at the end we will get the final 16 hidden features of all 20 inputs.
After seeing all 20 numbers, hidden state might be:
[0.95,    # "Trend: strongly increasing"
 0.87,    # "Pattern: arithmetic progression"
 0.12,    # "Anomalies: none detected"
 0.45,    # "Average value: medium-high"
 0.78,    # "Consistency: very consistent"
 -0.23,   # "Noise level: low"
 0.91,    # "Predictability: highly predictable"
 ... and 9 more abstract features]
It has summarized 20 numbers into just 16 numbers of "understanding"!

Visual Comparison: Memory vs Understanding

┌─────────────────────────────────────────────────────────┐
│  ❌ What People THINK:                                  
│                                                          
│  Hidden State = [1, 2, 3, 4, 5, 6, 7, 8, 9,10,11,12,13,14,15,16]
│                  ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑ ↑  ↑  ↑  ↑  ↑  ↑
│                  Just stored the last 16 inputs!        
│                                                          
├─────────────────────────────────────────────────────────┤
│  ✅ What It ACTUALLY Is:                                 
│                                                          
│  Hidden State = [0.95, 0.87, 0.12, 0.45, 0.78, -0.23, 0.91, ...]
│                  ↑      ↑      ↑     ↑     ↑      ↑      ↑
│                  Trend  Pattern No     Avg   Consis- Noise Pre-
│                  detec- detec- anom-  value tency   level dict-
│                  tor    tor    alies                ability
│                                                          
│  These are ABSTRACT FEATURES learned during training!   
└─────────────────────────────────────────────────────────┘
The Magic: Compression
Think of it like this:

20 raw numbers ──► RNN ──► 16 abstract features
[1,2,3,...,20]         [0.95, 0.87, 0.12, ..., 0.91]

The RNN learned to COMPRESS the sequence into meaningful features!
Real-World Analogy:

Reading a 500-page book:
- You don't memorize every word (that's 500,000 words!)
- You extract key themes, plot points, character developments
- Maybe 50 key insights summarize the entire book

RNN does the same: Compresses long sequences into a fixed-size "understanding"
Mathematical Perspective

Hidden State Update:
h_t = tanh(W·x_t + U·h_{t-1})

h_t size = 16 (always!)
x_t size = 1 (one number at a time)
h_{t-1} size = 16 (previous understanding)

So:
- Input: 1 number
- Previous understanding: 16 numbers
- Output: New 16-number understanding

The 16 numbers in h_t are a BLEND of:
- What the new input means (W·x_t)
- What we understood before (U·h_{t-1})

Like updating your understanding of a book with each new page!
Experiment: Hidden Size vs Sequence Length
python
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In Summary:
Aspect	                Explanation
Hidden size 16	        The RNN has 16 different "feature detectors"
It does NOT mean	    It can remember 16 previous inputs
What it CAN do	        Compress ANY length sequence into 16 meaningful numbers
Analogy	                Like summarizing a book into 16 key themes, not memorizing 16 pages
Why we need it	        To capture multiple aspects of the sequence simultaneously
Think of hidden state as the RNN's "understanding capacity" - how many different things it can notice about your sequence at once!

### Step 4 — Initialize Model

In [95]:
model = SimpleRNN()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

### Step 5 — Training Loop

In [ ]:
epochs = 10000

for epoch in range(epochs):

    output = model(X)                        # shape of output will be (96, 1) which is the predicted next number for each of the 96 sequences in X, and we will compare this with the actual target values in y to calculate the loss.
    
    loss = criterion(output.squeeze(), y)    # shape of output will be (96, 1) and shape of y will be (96,) so we need to squeeze the output to make it have the same shape as y before calculating the loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 2000 == 0:
        print("Epoch:",epoch,"Loss:",loss.item())

Epoch: 0 Loss: 0.0008204569458030164
Epoch: 2000 Loss: 0.03686339035630226
Epoch: 4000 Loss: 0.0007864158251322806
Epoch: 6000 Loss: 0.0007506093825213611
Epoch: 8000 Loss: 0.1258743554353714


### Step 6 — Test Prediction

In [ ]:
test = torch.tensor([[92,93,94]]).float().unsqueeze(-1)       # shape of test will be (1, 3, 1), we have one sequence of 3 numbers (92, 93, 94) and we want to predict the next number in this sequence, which should be 95. We add an extra dimension at the end to represent the feature dimension, since we have only one feature (the numbers themselves), so the shape becomes (1, 3, 1) which is compatible with the expected input shape for RNNs in PyTorch.

prediction = model(test)                 # shape of prediction will be (1, 1) which is the predicted next number in the sequence (92, 93, 94), we will compare this with the actual next number which is 95 to see how well our model has learned to predict the next number in the sequence.

print("Predicted:", prediction.item())

Predicted: 95.17268371582031
